In [1]:
%pip install azure-eventhub

StatementMeta(, aa8e9c18-fec9-472b-b7cf-ad4c2924dcf4, 7, Finished, Available, Finished, True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 13.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [ ]:
import requests
import json
from azure.eventhub import EventHubProducerClient, EventData
from datetime import datetime

# Config
APP_KEY = "your app key"
CONNECTION_STRING = "your connection string"
EVENTHUB_NAME = "esehln36p6ujw33tkvza60_eh"

# Fetch TfL data
url = f"https://api.tfl.gov.uk/Line/Mode/tube/Status?app_key={APP_KEY}"
response = requests.get(url)
raw_data = response.json()

# Send to Eventstream
producer = EventHubProducerClient.from_connection_string(
    conn_str=CONNECTION_STRING,
    eventhub_name=EVENTHUB_NAME
)

with producer:
    event_data_batch = producer.create_batch()
    for line in raw_data:
        # Safely get nested fields
        statuses = line.get("lineStatuses", [])
        status = statuses[0] if statuses else {}
        
        validities = status.get("validityPeriods", [])
        validity = validities[0] if validities else {}
        
        disruption = status.get("disruption") or {}

        event = {
            "line_id": line.get("id"),
            "line_name": line.get("name"),
            "mode_name": line.get("modeName"),
            "status_severity": status.get("statusSeverity"),
            "status": status.get("statusSeverityDescription"),
            "reason": status.get("reason"),
            "disruption_from": validity.get("fromDate"),
            "disruption_to": validity.get("toDate"),
            "is_active_disruption": validity.get("isNow", False),
            "disruption_category": disruption.get("category"),
            "closure_text": disruption.get("closureText"),
            "timestamp": datetime.utcnow().isoformat()
        }
        event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)

print(f"✅ {len(raw_data)} events sent to Eventstream!")

StatementMeta(, aa8e9c18-fec9-472b-b7cf-ad4c2924dcf4, 9, Finished, Available, Finished, False)

✅ 11 events sent to Eventstream!
